This file helps to create a csv of the entire dataset with no dropped columns with house area and region for each origin and destination. 

In [ ]:
import pandas as pd

In [ ]:
bus_data = pd.read_csv("../../data/A0008D - v_q_bus_stop (full).csv")
bus_location_data = pd.read_csv("../../data/correct_bus_location.csv")
bus_additional_data = pd.read_csv("../../data/A0008D - v_q_vls_marker (full).csv")

In [ ]:
bus_consolidated = bus_data.merge(
    bus_location_data,
    how='outer',
    left_on='BUS_STOP_CD',
    right_on='BusStopCode'
)

bus_needed = bus_consolidated[[
    "BUS_STOP_CD",
    "BUS_STOP_NAM",
    "MRK_ID_NUM",
    "BusStopCode",
    "Description",
    "Latitude",
    "Longitude"
]].copy()

bus_needed = bus_needed.rename(columns={
    "BUS_STOP_NAM": "STATION_NAME",
    "Latitude": "LATITUDE",
    "Longitude": "LONGITUDE"
})

bus_needed = bus_needed.rename(columns={
    "BUS_STOP_NAM": "STATION_NAME",
    "Description": "LTA_DESCRIPTION",
    "Latitude": "LATITUDE",
    "Longitude": "LONGITUDE"
})
bus_needed["FINAL_STOP_CODE"] = bus_needed["BUS_STOP_CD"].fillna(bus_needed["BusStopCode"])
bus_needed["STATION_NAME"] = bus_needed["STATION_NAME"].fillna(bus_needed["LTA_DESCRIPTION"])
bus_needed['Travel_Type'] = 'BUS'

bus_needed_final = bus_needed[[
    "FINAL_STOP_CODE",
    "STATION_NAME",
    "MRK_ID_NUM",
    "LATITUDE",
    "LONGITUDE",
    "Travel_Type"
]].copy()

bus_additional_data["LATITUDE"] = bus_additional_data["LATITUDE_VAL"] / 3600000
bus_additional_data["LONGITUDE"] = bus_additional_data["LONGITUDE_VAL"] / 3600000

bus_additional_data_clean = bus_additional_data[[
    "MRK_ID_NUM",
    "LATITUDE",
    "LONGITUDE"
]].copy()

bus_needed_final = bus_needed_final.merge(
    bus_additional_data_clean,
    on="MRK_ID_NUM",
    how="left",
    suffixes=("", "_additional")
)


bus_needed_final["LATITUDE"] = bus_needed_final["LATITUDE"].fillna(
    bus_needed_final["LATITUDE_additional"]
)

bus_needed_final["LONGITUDE"] = bus_needed_final["LONGITUDE"].fillna(
    bus_needed_final["LONGITUDE_additional"]
)

bus_needed_final = bus_needed_final.drop(columns=[
    "LATITUDE_additional",
    "LONGITUDE_additional"
])

print("Remaining missing coords:",
      bus_needed_final["LATITUDE"].isna().sum())

In [ ]:
print(bus_consolidated.columns)

In [ ]:
train_data = pd.read_csv('../../data/mrt_station_coordinates.csv')
train_needed = train_data[['NODE_NAME', 'NODE_MRK_ID', 'LATITUDE', 'LONGITUDE']]
train_needed = train_needed.rename(columns={
    'NODE_NAME': 'STATION_NAME',
    'NODE_MRK_ID': 'MRK_ID_NUM'
})
train_needed['Travel_Type'] = 'TRAIN'
#train_needed.head(5)

In [ ]:
bus_needed_final = bus_needed_final[[
    "STATION_NAME",
    "MRK_ID_NUM",
    "LATITUDE",
    "LONGITUDE",
    "Travel_Type"
]]

consolidated_stations = pd.concat(
    [bus_needed_final, train_needed],
    axis=0,
    ignore_index=True
)

consolidated_stations.tail(5)

In [ ]:
df = pd.read_csv('../../data/A0008D - v_pt_ride_all (full).csv')

In [ ]:
df['BIZ_DT'] = pd.to_datetime(df['BIZ_DT'])
df['ENTRY_DT'] = pd.to_datetime(df['ENTRY_DT'])
df['EXIT_DT'] = pd.to_datetime(df['EXIT_DT'])
df['ENTRY_TM'] = pd.to_datetime(
   df['ENTRY_DT'].astype(str) + ' ' + df['ENTRY_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
df['EXIT_TM'] = pd.to_datetime(
   df['EXIT_DT'].astype(str) + ' ' + df['EXIT_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)


In [ ]:
df.dtypes

In [ ]:
df = df.merge(consolidated_stations, how = 'left', left_on= 'DEST_LOC_ID_NUM',
                right_on= 'MRK_ID_NUM')

In [ ]:
df.rename(columns={
    'STATION_NAME': 'DEST_STATION_NAME',
    'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
    'LATITUDE': 'DEST_LATITUDE',
    'LONGITUDE': 'DEST_LONGITUDE',
    'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [ ]:
df = df.merge(consolidated_stations, how = 'left', left_on= 'ORIG_LOC_ID_NUM',
                right_on= 'MRK_ID_NUM')

In [ ]:
df.rename(columns={
    'STATION_NAME': 'ORIG_STATION_NAME',
    'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
    'LATITUDE': 'ORIG_LATITUDE',
    'LONGITUDE': 'ORIG_LONGITUDE',
    'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)


In [ ]:
abt = pd.read_csv('../../data/A0008D - v_abt_pt_ride (full).csv')

In [ ]:
abt['BIZ_DT'] = pd.to_datetime(abt['BIZ_DT'])
abt['ENTRY_DT'] = pd.to_datetime(abt['ENTRY_DT'])
abt['EXIT_DT'] = pd.to_datetime(abt['EXIT_DT'])
abt['ENTRY_TM'] = pd.to_datetime(
   abt['ENTRY_DT'].astype(str) + ' ' + abt['ENTRY_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
abt['EXIT_TM'] = pd.to_datetime(
   abt['EXIT_DT'].astype(str) + ' ' + abt['EXIT_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)

In [ ]:
abt2 = abt.merge(consolidated_stations, how = 'left', left_on= 'DEST_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')
#abt2.head(5)

In [ ]:
abt2.rename(columns={
   'STATION_NAME': 'DEST_STATION_NAME',
   'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
   'LATITUDE': 'DEST_LATITUDE',
   'LONGITUDE': 'DEST_LONGITUDE',
   'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [ ]:
abt2 = abt2.merge(consolidated_stations, how = 'left', left_on= 'ORIG_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [ ]:
abt2.rename(columns={
   'STATION_NAME': 'ORIG_STATION_NAME',
   'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
   'LATITUDE': 'ORIG_LATITUDE',
   'LONGITUDE': 'ORIG_LONGITUDE',
   'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)


In [ ]:
abt2.shape

Combine pt_ride and abt_ride

In [ ]:
df['CRD_NUM'] = df['CRD_NUM'].astype('object')


In [ ]:
df.dtypes

In [ ]:
df2 = pd.concat(
   [df, abt2],
   axis=0,
   ignore_index=True
)


#df2.tail(5)

In [ ]:
df2.shape

In [ ]:
df3 = df2[df2["ENTRY_DT"].dt.date == pd.Timestamp("2025-02-12").date()]
df3 = df3.sort_values(["CRD_NUM", "ENTRY_DT", "ENTRY_TM"], ascending= [True, True, True]).reset_index(drop= True)

In [ ]:
# Onemap API does not allow dates more than a year
df3["ENTRY_DT"] = pd.Timestamp("10-04-2025")
df3["EXIT_DT"] = pd.Timestamp("10-04-2025")
df3["ENTRY_TM"] = df3["ENTRY_TM"].dt.strftime("%H:%M:%S")
df3["EXIT_TM"] = df3["EXIT_TM"].dt.strftime("%H:%M:%S")
df3["ENTRY_DT"] = df3["ENTRY_DT"].dt.strftime("%m-%d-%Y")
df3["EXIT_DT"] = df3["EXIT_DT"].dt.strftime("%m-%d-%Y")

In [ ]:
df3["next_orig_lat"] = df3.groupby("CRD_NUM")["ORIG_LATITUDE"].shift(-1)
df3["next_orig_lon"] = df3.groupby("CRD_NUM")["ORIG_LONGITUDE"].shift(-1)
df3["next_orig_station"] = df3.groupby("CRD_NUM")["ORIG_STATION_NAME"].shift(-1)

In [ ]:
walk_cache = pd.read_csv('../../data/walk_distance_cache.csv')

In [ ]:
df3["pair_key"] = (
  df3["DEST_LATITUDE"].astype(str) + "_" +
  df3["DEST_LONGITUDE"].astype(str) + "_" +
  df3["next_orig_lat"].astype(str) + "_" +
  df3["next_orig_lon"].astype(str)
)
pairs = df3[
   [
       "pair_key",
       "DEST_STATION_NAME",
       "next_orig_station",
       "DEST_LATITUDE",
       "DEST_LONGITUDE",
       "next_orig_lat",
       "next_orig_lon"
   ]
].dropna(subset=[
   "DEST_LATITUDE",
   "DEST_LONGITUDE",
   "next_orig_lat",
   "next_orig_lon"
]).drop_duplicates("pair_key").copy()


In [ ]:
df3 = (
   df3.drop(columns=["pair_key"], errors="ignore")
   .merge(
       walk_cache.drop(columns=["pair_key"], errors="ignore")[[
           "DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon", "walk_distance"
       ]].drop_duplicates(
           subset=["DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon"]
       ),
       on=["DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon"],
       how="left"
   )
)

In [ ]:
# restore the actual filtered business date
real_date = pd.Timestamp("2025-02-12")


df3["ENTRY_DT"] = real_date
df3["EXIT_DT"] = real_date


# if ENTRY_TM / EXIT_TM are currently strings like '00:47:26',
# convert them back into full datetimes
df3["ENTRY_TM"] = pd.to_datetime(
   df3["ENTRY_DT"].astype(str) + " " + df3["ENTRY_TM"].astype(str),
   format="%Y-%m-%d %H:%M:%S",
   errors="coerce"
)


df3["EXIT_TM"] = pd.to_datetime(
   df3["EXIT_DT"].astype(str) + " " + df3["EXIT_TM"].astype(str),
   format="%Y-%m-%d %H:%M:%S",
   errors="coerce"
)

Getting planning area dataset
- API is from https://www.onemap.gov.sg/apidocs/planningarea/#planningAreaPolygon for year 2019 which is the latest available year
- Please get your own token from One Map and insert it into headers and uncomment that line

In [ ]:
#please get your own token from one map api and insert into headers and uncomment headers. 
import requests
    
url = "https://www.onemap.gov.sg/api/public/popapi/getAllPlanningarea?year=2019"
    
#headers = {"Authorization": "****insert ur token here***"}
    
response = requests.request("GET", url, headers=headers)
    
print(response.text)

data = response.json()

In [ ]:
print(type(data))
print(data.keys())

In [ ]:
records = data['SearchResults']
areas = pd.DataFrame(records)
import json
areas['geojson_parsed'] = areas['geojson'].apply(json.loads)

In [ ]:
from shapely.geometry import shape
import geopandas as gpd

areas['geometry'] = areas['geojson_parsed'].apply(shape)

gdf = gpd.GeoDataFrame(areas, geometry='geometry', crs="EPSG:4326")

In [ ]:
gdf = gdf[['pln_area_n', 'geometry']]
gdf = gdf.rename(columns={'pln_area_n': 'planning_area'})

In [ ]:
#gdf.head(20)

In [ ]:
def get_planning_area(df, lat_col, lon_col):
    gdf_points = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326"
    )
    return gpd.sjoin(gdf_points, gdf, how="left", predicate="within")['planning_area'].values

In [ ]:
first_tap = df3.groupby('CRD_NUM', as_index=False).first()
first_tap['house_area'] = get_planning_area(first_tap, 'ORIG_LATITUDE', 'ORIG_LONGITUDE')

df3 = df3.merge(first_tap[['CRD_NUM', 'house_area']], on='CRD_NUM', how='left')

In [ ]:
# find origin and destination for every row
df3['orig_region'] = get_planning_area(df3, 'ORIG_LATITUDE', 'ORIG_LONGITUDE')
df3['dest_region'] = get_planning_area(df3, 'DEST_LATITUDE', 'DEST_LONGITUDE')

In [ ]:
#verification
#see the new columns
print(df3[['CRD_NUM', 'ORIG_STATION_NAME', 'orig_region', 
           'DEST_STATION_NAME', 'dest_region', 'house_area']].head(20))

# check for NaNs: should be woodlands checkpoint only
print("orig_region NaNs:", df3['orig_region'].isna().sum())
print("dest_region NaNs:", df3['dest_region'].isna().sum())
print("house_area NaNs:", df3['house_area'].isna().sum())

# pick a station you know and verify the region looks right
print(df3[df3['ORIG_STATION_NAME'].str.contains('BISHAN', na=False)]
      [['ORIG_STATION_NAME', 'orig_region']].drop_duplicates())

print(df3[df3['DEST_STATION_NAME'].str.contains('ORCHARD', na=False)]
      [['DEST_STATION_NAME', 'dest_region']].drop_duplicates())

# check distribution of regions
print(df3['orig_region'].value_counts())
print(df3['dest_region'].value_counts())

# check house area: each card should have exactly one house_area
print(df3.groupby('CRD_NUM')['house_area'].nunique().value_counts())
# should only show 1, meaning every card has exactly one house_area

In [ ]:
df3.to_pickle("all_df_regions.pkl")
df3 = pd.read_pickle("master_df_regions.pkl")

In [ ]:
#save as csv file
df3.to_csv("all_df_regions", index=False)